# 設備異常告警：不平衡分類與防資料洩漏

本 lab 使用 2,000 列合成資料，不含個資。請先讀三個學習單元；Colab 使用者需手動上傳 CSV 與 split manifest。

In [ ]:
from pathlib import Path
import copy
import hashlib
import json
import math
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_SHA256 = "sha256:c6db9b752a39e85e48ddf13131eafc31ca1320c74bb171582ef49d8a58ad41ee"
SPLIT_SHA256 = "sha256:92a43c167af21d254f0b07ff0a1386c48bbe0d7ea3ef8f12a4ca588d7681882e"

def locate(filename):
    candidates = [Path("fixtures") / filename, Path(filename), Path("notebooks/labs/lab-imbalanced-classification/fixtures") / filename]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"找不到 {filename}；請把 fixture CSV 與 split manifest 上傳到目前工作目錄或 fixtures/。")

def file_hash(path):
    return "sha256:" + hashlib.sha256(path.read_bytes()).hexdigest()

data_path = locate("equipment_alerts_v1.csv")
split_path = locate("split_manifest_v1.json")
assert file_hash(data_path) == DATA_SHA256, "fixture checksum 不符"
assert file_hash(split_path) == SPLIT_SHA256, "split manifest checksum 不符"
print({"python": sys.version.split()[0], "pandas": pd.__version__, "sklearn": sklearn.__version__})


In [ ]:
data = pd.read_csv(data_path)
split_manifest = json.loads(split_path.read_text(encoding="utf-8"))
assert data.shape == (2000, 10)
assert [c for c in data.columns if c.startswith("feature_")] == [f"feature_{i}" for i in range(1, 9)]
assert int(data["is_anomaly"].sum()) == 100
split_ids = {name: set(ids) for name, ids in split_manifest["splits"].items()}
assert {name: len(ids) for name, ids in split_ids.items()} == {"train": 1200, "validation": 400, "test": 400}
assert not (split_ids["train"] & split_ids["validation"])
assert not (split_ids["train"] & split_ids["test"])
assert not (split_ids["validation"] & split_ids["test"])
assert set(data["row_id"]) == set().union(*split_ids.values())
parts = {name: data[data["row_id"].isin(ids)].sort_values("row_id").reset_index(drop=True) for name, ids in split_ids.items()}
feature_cols = [f"feature_{i}" for i in range(1, 9)]


## Worked example

先預測 accuracy 是否勝過全負類 baseline，再執行手算驗證。矩陣排列固定為 [[TN, FP], [FN, TP]]。

In [ ]:
def metrics_from_counts(tn, fp, fn, tp):
    total = tn + fp + fn + tp
    accuracy = (tp + tn) / total
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

def metrics_from_predictions(y_true, predictions, fn_cost, fp_cost):
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    metrics = metrics_from_counts(int(tn), int(fp), int(fn), int(tp))
    metrics.update({"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp), "average_cost": (fn_cost * fn + fp_cost * fp) / len(y_true)})
    return metrics

worked = metrics_from_counts(tn=362, fp=18, fn=8, tp=12)
expected = {"accuracy": 0.935, "precision": 0.4, "recall": 0.6, "f1": 0.48}
assert all(abs(worked[key] - value) <= 1e-6 for key, value in expected.items())
assert 380 / 400 == 0.95  # 全負類 baseline；其異常 recall 為 0
worked


## Guided practice

先確認所有 fit row IDs 都屬 train，再比較 DummyClassifier 與 LogisticRegression。Pipeline 管理轉換 fit 順序，但仍要人工排除預測當下不存在的欄位。

In [ ]:
def make_model():
    preprocessor = ColumnTransformer([("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_cols)])
    return Pipeline([("preprocess", preprocessor), ("classifier", LogisticRegression(max_iter=500, random_state=42))])

def fit_with_audit(estimator, frame):
    """Fit from the supplied frame and return the row IDs actually observed."""
    observed_row_ids = sorted(frame["row_id"].tolist())
    estimator.fit(frame[feature_cols], frame["is_anomaly"])
    return observed_row_ids

model = make_model()
dummy = DummyClassifier(strategy="most_frequent")
train = parts["train"]
validation = parts["validation"]
test = parts["test"]
fit_row_ids = fit_with_audit(model, train)
dummy_fit_row_ids = fit_with_audit(dummy, train)
validation_scores = model.predict_proba(validation[feature_cols])[:, 1]
assert set(fit_row_ids) == split_ids["train"]
assert dummy_fit_row_ids == fit_row_ids


## Independent task

以 validation 選 threshold，FN 成本 20、FP 成本 1；同成本依 recall 高、threshold 小排序。凍結後只讀一次 test。再把 FN 成本改成 5 做敏感度分析。

In [ ]:
THRESHOLDS = np.round(np.arange(0.05, 1.0, 0.05), 2)

def evaluate_scores(y_true, scores, threshold, fn_cost, fp_cost):
    predictions = (np.asarray(scores) >= threshold).astype(int)
    metrics = metrics_from_predictions(y_true, predictions, fn_cost, fp_cost)
    metrics["threshold"] = float(threshold)
    return metrics, predictions.tolist()

def select_threshold(frame, scores, fn_cost=20, fp_cost=1):
    """Select from the supplied frame and return the row IDs actually observed."""
    assert len(frame) == len(scores), "scores 與選擇資料列數不一致"
    rows = [evaluate_scores(frame["is_anomaly"], scores, threshold, fn_cost, fp_cost)[0] for threshold in THRESHOLDS]
    selected = min(rows, key=lambda row: (row["average_cost"], -row["recall"], row["threshold"]))
    return selected, rows, sorted(frame["row_id"].tolist())

selected, validation_table, threshold_selection_row_ids = select_threshold(validation, validation_scores)
sensitivity_selected, sensitivity_table, sensitivity_row_ids = select_threshold(validation, validation_scores, fn_cost=5, fp_cost=1)
frozen_threshold = float(selected["threshold"])

# The test split is touched only after both validation decisions are complete and frozen.
test_scores = model.predict_proba(test[feature_cols])[:, 1]
test_result, test_predictions = evaluate_scores(test["is_anomaly"], test_scores, frozen_threshold, 20, 1)
dummy_predictions = dummy.predict(test[feature_cols]).astype(int).tolist()
dummy_result = metrics_from_predictions(test["is_anomaly"], dummy_predictions, 20, 1)

result = {
    "schemaVersion": 1,
    "dataChecksum": DATA_SHA256,
    "splitHash": SPLIT_SHA256,
    "seed": 42,
    "modelConfig": {"estimator": "LogisticRegression", "max_iter": 500, "random_state": 42},
    "fitRowIds": fit_row_ids,
    "dummyFitRowIds": dummy_fit_row_ids,
    "thresholdSelectionRowIds": threshold_selection_row_ids,
    "threshold": frozen_threshold,
    "costs": {"fn": 20, "fp": 1},
    "validationTable": validation_table,
    "test": {**test_result, "rowIds": test["row_id"].tolist(), "yTrue": test["is_anomaly"].astype(int).tolist(), "yPred": test_predictions,
             "dummy": {**dummy_result, "yPred": dummy_predictions}},
    "sensitivityFn5": {"threshold": sensitivity_selected["threshold"], "thresholdSelectionRowIds": sensitivity_row_ids, "validationTable": sensitivity_table},
}
Path("metrics.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
Path("decision.md").write_text("# 試點決策草稿\n\n此結果來自合成資料，只驗證流程；請依 metrics.json 填入方案比較、限制與試點下一步。\n", encoding="utf-8")
selected, test_result


## Solution note

完整解答會生成 `metrics.json` 與 `decision.md`。數字來自固定合成 fixture，不代表真實設備績效。

In [ ]:
def assert_result(candidate):
    assert set(candidate["fitRowIds"]) == split_ids["train"], "fit rows 必須精確等於 train"
    assert set(candidate["dummyFitRowIds"]) == split_ids["train"], "Dummy fit rows 必須精確等於 train"
    assert set(candidate["thresholdSelectionRowIds"]) == split_ids["validation"], "threshold 只能用 validation"
    assert set(candidate["sensitivityFn5"]["thresholdSelectionRowIds"]) == split_ids["validation"], "敏感度分析只能用 validation"
    payload = candidate["test"]
    assert payload["rowIds"] == test["row_id"].tolist(), "test row IDs 必須精確對齊固定 split"
    assert payload["yTrue"] == test["is_anomaly"].astype(int).tolist(), "test labels 必須對齊 fixture"
    tn, fp, fn, tp = confusion_matrix(payload["yTrue"], payload["yPred"], labels=[0, 1]).ravel()
    assert [int(tn), int(fp), int(fn), int(tp)] == [payload[k] for k in ("tn", "fp", "fn", "tp")]
    recalculated = metrics_from_counts(int(tn), int(fp), int(fn), int(tp))
    for key in ("accuracy", "precision", "recall", "f1"):
        assert math.isfinite(payload[key]) and 0 <= payload[key] <= 1
        assert abs(payload[key] - recalculated[key]) <= 1e-6, f"{key} 與 matrix 不一致"
    expected = min(candidate["validationTable"], key=lambda row: (row["average_cost"], -row["recall"], row["threshold"]))
    assert candidate["threshold"] == expected["threshold"], "threshold 未依 validation 成本與 tie-break 選擇"
    assert payload["dummy"]["yPred"] == dummy.predict(test[feature_cols]).astype(int).tolist(), "Dummy predictions 不一致"

assert_result(result)

def must_reject(mutator):
    broken = copy.deepcopy(result)
    mutator(broken)
    try:
        assert_result(broken)
    except AssertionError:
        return True
    raise AssertionError("fault injection 未被檢查攔下")

# Faults execute the wrong operation first; the audit captures its real input rows.
leaky_model = make_model()
leaky_fit_row_ids = fit_with_audit(leaky_model, data)
assert must_reject(lambda x: x.__setitem__("fitRowIds", leaky_fit_row_ids))
_, _, test_selection_row_ids = select_threshold(test, test_scores)
assert must_reject(lambda x: x.__setitem__("thresholdSelectionRowIds", test_selection_row_ids))
assert must_reject(lambda x: x["test"].update({"precision": x["test"]["recall"], "recall": x["test"]["precision"]}))
print("L3 checks 與 3 個 fault injections 全部通過")


## Reflection

回答：正類比例改變會影響什麼？低成本是否足以支持上線？若同設備有多次觀測，切分契約要如何改？